# fusion_run — chạy pipeline trên rổ ứng viên FUSION của D

Thay `bm25_top100_*` bằng `fusion_rrf_top50_*`. Đo trên dev300 đã cho (CPU, 0 giờ GPU):

| | BM25 (đang dùng) | Fusion |
|---|---|---|
| recall@5 thô | 0.7533 | **0.8733** |
| recall@50 | 0.9683 | **0.9817** |
| trần | 0.9783 *(100 văn bản)* | **0.9817 *(50 văn bản)*** |

Trần dâng lên **và** rổ nhỏ đi một nửa → tầng 1 rẻ đi một nửa.

**Ngưỡng đặt TRƯỚC khi chạy** (mốc hiện tại `max n=2` = 0.9183):

| ra | quyết định |
|---|---|
| ≥ 0.9383 | thắng rõ → chạy đề thi, nộp |
| 0.9283 – 0.9383 | khớp mô phỏng → vẫn chạy đề thi (đổi **cơ chế**, không phải vặn tham số) |
| 0.9183 – 0.9283 | thấp hơn cả chặn dưới → nghi lỗi đọc rổ mới |
| **< 0.9183** | **CHẮC CHẮN BUG.** Trần đã dâng thì không thể tụt. Dừng, soi lại |

**`rrf_score` được nạp vào khoá `bm25`** để `blend_bm25_first` và `DC.rank_by` dùng lại
nguyên xi, không phải sửa file dùng chung nào. Tên khoá sai nhưng nội dung đúng.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# ===== Bước 0: cấu hình + đường dẫn + dấu vân tay =====
import os, sys, json, time, hashlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODE = "public_k50"   # <-- lượt K=50. Xem Q_TU/Q_DEN bên dưới.
# Vì sao K=50: đo 28/08 trên 24 cặp (câu, văn-bản-gold) có oracle đủ đoạn —
#   đoạn TỐT NHẤT nằm trong top-20 BM25 ở 20/24 cặp (K=20 bắt 90% giá trị oracle).
#   4 cặp hở ở hạng 23, 23, 50, 109 -> K=50 bắt thêm 3/4. Trần ước ~+0,3 điểm dev,
#   và đó là CHẶN TRÊN (mẫu toàn câu đang sai). Khớp quét K 19/08: +0,80 ở K=120.
RO   = "_matchEmbedded"   # "" = rổ fusion cũ · "_matchEmbedded" = rổ D giao 23/08
TAG  = "matchEmb_K50"          # tên NGẮN dán vào file nộp. Chạy public_b thì đổi thành "matchEmb_M20"
                          # rổ mới: recall@50 0.9817->0.9883 · recall@5 thô +3,83
M_DOC, K_CHUNK, TOPK = 20, 50, 5      # K: 20 -> 50
Q_TU, Q_DEN = 0, 1000                 # lát câu hỏi. Bước 3 ước >6h thì chia đôi:
                                      # lượt 1 (0,500) -> tải outputs/ lên dataset -> lượt 2 (500,1000)
MOC_CU = 0.9350       # max n=1 trên rổ fusion CŨ — mốc phải vượt
NS = (0, 1, 2, 3, 4)  # rổ đổi thì ĐỈNH n DỜI (kết luận 3, 23/08). Quét trọn, 0 giờ GPU

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

CFG = {                       # (file ứng viên, file câu hỏi, skip, tên file ra)
 "dev":      (f"{INPUT_DIR}/fusion_rrf_top50_dev_1000{RO}.json", f"{INPUT_DIR}/dev_300_locked.json",     0,  f"scores_dev300_fusion_M20_K20{RO}.json"),
 "public_k50":(f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",  f"{INPUT_DIR}/public-official.json",     0,  f"scores_public_fusion_M20_K50{RO}.json"),
 "public_a": (f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",   f"{INPUT_DIR}/public-official.json",     0,  f"scores_public_fusion_M10_K20{RO}.json"),
 "public_b": (f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",   f"{INPUT_DIR}/public-official.json",    10,  f"scores_public_fusion_M20_K20{RO}.json"),
 "public_t1":(f"{INPUT_DIR}/fusion_rrf_top50_public{RO}.json",   f"{INPUT_DIR}/public-official.json",     0,  f"scores_public_fusion_TANG1{RO}.json"),   # CHỈ tầng 1, có checkpoint
}
CAND_F, Q_F, SKIP, OUT_NAME = CFG[MODE]
M_RUN = 10 if MODE == "public_a" else M_DOC      # đề thi tách 2 lượt cho dưới trần 12h

for f in ("deep_chunk.py", "rerank_from_d.py", "rerank.py"):
    b = open(f"{INPUT_DIR}/{f}", "rb").read()
    print(f"{f:22} {len(b):>6} bytes  {hashlib.sha256(b).hexdigest()[:12]}")

import deep_chunk as DC
from rerank import load_reranker
from rerank_from_d import blend_bm25_first, score_docs_from_d
DC.MERGE_CHARS = 1800                      # cùng cấu hình bài chốt 0.8871

assert hasattr(DC, "MERGE_CHARS"), "deep_chunk.py là BẢN CŨ — upload lại"
assert os.path.isfile(CAND_F), f"CHƯA UPLOAD {CAND_F}"
print(f"\nMODE={MODE} · M={M_RUN} · SKIP={SKIP} · CTX={CTX_DIR}")

## Bước 1 — Nạp rổ fusion, kiểm khớp câu hỏi

`rrf_score` vào khoá `bm25`: `blend_bm25_first` sẽ ép 2 suất đầu theo thứ tự **fusion**
chứ không phải BM25 thô. Đó là chỗ ăn điểm — fusion xếp recall@5 = 0.8733 so với 0.7533.

In [ ]:
cand = json.load(open(CAND_F, encoding="utf-8"))
qsrc = json.load(open(Q_F, encoding="utf-8"))
questions = {q: v["question"] for q, v in qsrc.items() if q in cand}
assert len(questions) == len(qsrc), f"THIẾU {len(qsrc)-len(questions)} câu trong rổ ứng viên"

rrf = {q: {str(c["doc_id"]): float(c["rrf_score"]) for c in cand[q]} for q in questions}
qs_all = list(questions)
QLAT = {q: questions[q] for q in qs_all[Q_TU:Q_DEN]} if MODE == "public_k50" else dict(questions)
print(f"lát câu hỏi lượt này: {len(QLAT)}/{len(questions)}  [{Q_TU}:{Q_DEN}]")
nc = [len(cand[q]) for q in questions]
print(f"{len(questions)} câu · {min(nc)}-{max(nc)} ứng viên/câu")
print(f"tổng chunk tầng 1: {sum(len(c['top_chunks']) for q in questions for c in cand[q]):,}")

if MODE == "dev":
    gold = {q: {str(x) for x in qsrc[q]["answer"]} for q in questions}
    fr = {q: [str(c["doc_id"]) for c in cand[q]] for q in questions}
    for k in (5, 50):
        r = sum(len(gold[q] & set(fr[q][:k]))/len(gold[q]) for q in gold)/len(gold)
        print(f"  TRẦN rổ: recall@{k} = {r:.4f}" + ("   <- không ai vượt được" if k == 50 else ""))

## Bước 2 — Tầng 1: chấm `top_chunks` của D (rổ 50 nên rẻ bằng nửa lần trước)

In [ ]:
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda")

if MODE == "public_k50":
    # Tầng 1 KHÔNG phụ thuộc K -> tái dùng nguyên xi, tiết kiệm ~2h.
    # NHƯNG PHẢI BỎ `ce_deep` cũ: nó chấm ở K=20, đổi K thì không so được (cảnh báo trong deepen_one).
    RES = f"{INPUT_DIR}/{OUT_NAME}"                    # lượt K=50 TRƯỚC ĐÓ, nếu đã chạy nửa đầu
    if os.path.isfile(RES):
        scores = json.load(open(RES, encoding="utf-8"))
        xong = sum(1 for s in scores.values() if any("ce_deep" in v for v in s.values()))
        print(f"nạp lại lượt K=50 trước: {xong} câu đã đọc sâu, {len(scores)-xong} câu còn lại")
    else:
        SRC = f"{INPUT_DIR}/scores_public_fusion_M20_K20{RO}.json"
        assert os.path.isfile(SRC), f"CHƯA UPLOAD {SRC} — lượt K=50 tái dùng tầng 1 của lượt K=20"
        raw = json.load(open(SRC, encoding="utf-8"))
        scores = {q: {d: {"ce": v["ce"], "bm25": v["bm25"]} for d, v in s.items()} for q, s in raw.items()}
        assert len(scores) == len(questions), f"lệch số câu: {len(scores)} vs {len(questions)}"
        print(f"tái dùng tầng 1 K=20 cho {len(scores)} câu · ĐÃ BỎ ce_deep cũ · BỎ QUA Bước 2")
    # chỉ đọc sâu những câu trong lát MÀ CHƯA có ce_deep -> chạy lại là chạy tiếp
    QLAT = {q: v for q, v in QLAT.items() if not any("ce_deep" in x for x in scores[q].values())}
    print(f"cần đọc sâu lượt này: {len(QLAT)} câu")
else:
  # public_b nạp lại tầng 1 của public_a -> khỏi chấm lại ~100k đoạn (~2h)
  PREV = f"{INPUT_DIR}/scores_public_fusion_M10_K20{RO}.json"   # lượt B nạp output lượt A
  if MODE == "public_b" and os.path.isfile(PREV):
      scores = json.load(open(PREV, encoding="utf-8"))
      nd = [sum(1 for v in s.values() if "ce_deep" in v) for s in scores.values()]
      assert min(nd) == max(nd) == 10, f"file lượt A phải có ĐÚNG 10 ce_deep/câu, thấy {min(nd)}-{max(nd)}"
      print(f"nạp lại tầng 1 + ce_deep hạng 1-10 từ lượt A: {len(scores)} câu — BỎ QUA Bước 2")
  else:
      assert MODE != "public_b", f"CHƯA UPLOAD {PREV} — lượt B cần output của lượt A"
      p1 = f"{OUT}/tang1_{OUT_NAME}"
      # nối tiếp lượt trước nếu đã upload file dở (hết quota giữa chừng thì chạy lại là tiếp)
      RESUME = f"{INPUT_DIR}/tang1_{OUT_NAME}"
      scores = json.load(open(RESUME, encoding="utf-8")) if os.path.isfile(RESUME) else {}
      todo = [q for q in questions if q not in scores]
      print(f"tầng 1: đã có {len(scores)} câu, cần chạy {len(todo)} câu")

      t0 = time.time()
      for i, q in enumerate(todo, 1):
          ce = score_docs_from_d(questions[q], cand[q], score_fn)
          scores[q] = {d: {"ce": float(s), "bm25": rrf[q].get(d, 0.0)} for d, s in ce.items()}
          if i % 100 == 0 or i == len(todo):
              json.dump(scores, open(p1, "w", encoding="utf-8"), ensure_ascii=False)  # checkpoint
              el = time.time() - t0
              print(f"  {i}/{len(todo)} | {el/60:.1f} phút | còn ~{el/i*(len(todo)-i)/60:.1f} phút | đã lưu", flush=True)

      json.dump(scores, open(p1, "w", encoding="utf-8"), ensure_ascii=False)
      print(f"ĐÃ LƯU {p1} ({len(scores)}/{len(questions)} câu) — TẢI VỀ dù phần dưới hỏng")
      if MODE == "public_t1":
          print("\nMODE=public_t1: DỪNG Ở ĐÂY. Upload file trên lên dataset rồi chạy public_a.")
          raise SystemExit(0)

## Bước 3 — ĐẾM trước khi chấm tầng 2 (quy tắc 6)

Vượt 6h thì dừng: hạ `K_CHUNK` xuống 12, hoặc với đề thi thì tách `public_a` / `public_b`.

In [ ]:
t0 = time.time()
n2 = DC.count_deep_chunks(QLAT, scores, CTX_DIR, M_RUN, K_CHUNK, skip=SKIP)
print(f"tầng 2: {n2:,} đoạn ({n2/len(questions):.0f}/câu) | băm+đếm {time.time()-t0:.0f}s")
print(f"ước {n2/12.9/3600:.1f}h @12,9 đoạn/s  ·  {n2/6.5/3600:.1f}h nếu chậm 2x")
gio = n2/12.9/3600
assert gio < 9.5, (f"ước {gio:.1f}h — quá sát trần 12h. Chia đôi lát câu hỏi: "
                   f"đặt Q_TU,Q_DEN = 0,{len(QLAT)//2} rồi chạy lại.")
if gio > 6: print(f"CẢNH BÁO: ước {gio:.1f}h. Chậm 2x là chết. Cân nhắc chia đôi lát.")

## Bước 4 — Tầng 2: đọc sâu top-M theo hạng CE

In [ ]:
# deepen_all CHỈ trả về những câu được truyền vào -> phải MERGE, không gán đè,
# nếu không là mất tầng 1 của các câu ngoài lát.
# Chia lô 100 câu + lưu sau mỗi lô: lượt này ~8h, chết ở giờ thứ 7 mà không checkpoint
# thì mất trắng (đúng bài học lượt 15/08 bị trần 12h giết).
t0 = time.time(); lat = list(QLAT); p2 = f"{OUT}/{OUT_NAME}"
for j in range(0, len(lat), 100):
    lo = {q: QLAT[q] for q in lat[j:j+100]}
    scores.update(DC.deepen_all(lo, scores, CTX_DIR, score_fn, M_RUN, K_CHUNK, skip=SKIP))
    json.dump(scores, open(p2, "w", encoding="utf-8"), ensure_ascii=False)
    d = j + len(lo); el = time.time() - t0
    print(f"== {d}/{len(lat)} câu | {el/60:.0f} phút | còn ~{el/d*(len(lat)-d)/60:.0f} phút | ĐÃ LƯU {p2}", flush=True)

print(f"\ntầng 2 xong {(time.time()-t0)/60:.0f} phút")
print(f"ĐÃ LƯU {p2} — TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN")


## Bước 5 — Đo (dev) / đóng gói bài nộp (đề thi)

In [ ]:
order = {q: [d for d, _ in sorted(rrf[q].items(), key=lambda x: -x[1])] for q in questions}
pred = lambda v, n: {q: blend_bm25_first(DC.rank_by(scores[q], v), order[q], k=TOPK, n_bm25=n)
                     for q in questions}

if MODE == "dev":
    rec = lambda v, n: sum(len(gold[q] & set(p))/len(gold[q]) for q, p in pred(v, n).items())/len(gold)
    print(f"{'biến thể':10s}" + "".join(f"  n={n}   " for n in NS))
    tab = {}
    for v in DC.VARIANTS:
        tab[v] = {n: rec(v, n) for n in NS}
        print(f"{v:10s}" + "".join(f"  {tab[v][n]:.4f}" for n in NS))
    got = tab["max"][2]
    print(f"\nmax n=2 = {got:.4f} | mốc rổ cũ {MOC_CU} | Δ {(got-MOC_CU)*100:+.2f} điểm")
    print("mô phỏng CPU trước đó dự báo 0.9283 (chặn dưới)")
    print("=" * 60)
    if got >= 0.9383:   print("THẮNG RÕ -> chạy public_a rồi public_b, nộp")
    elif got >= 0.9283: print("KHỚP MÔ PHỎNG -> vẫn chạy đề thi, đây là đổi CƠ CHẾ")
    elif got >= MOC_CU: print("THẤP HƠN CHẶN DƯỚI -> soi lại cách nạp rrf/thứ tự trước khi chạy tiếp")
    else:               print("!!! TỤT DƯỚI MỐC = BUG. Trần đã dâng thì không thể tụt. DỪNG.")
else:
    thieu = [q for q in questions if not any("ce_deep" in v for v in scores[q].values())]
    if thieu:
        print(f"CHƯA ĐỦ: còn {len(thieu)} câu chưa đọc sâu -> KHÔNG đóng gói bài nộp.")
        print(f"Tải outputs/{OUT_NAME} lên dataset rồi chạy lượt sau với Q_TU,Q_DEN = {Q_DEN},1000")
        raise SystemExit(0)
    import zipfile
    dev1k = json.load(open(f"{INPUT_DIR}/dev_1000_locked.json", encoding="utf-8"))
    for n in NS:
        sub = {q: {"answer": list(p)} for q, p in pred("max", n).items()}
        assert len(sub) == 1000 and set(sub) == set(qsrc), "qid không khớp đề thi"
        assert not (set(sub) & set(dev1k)), "LỌT CÂU DEV — dừng"
        assert all(len(v["answer"]) == 5 == len(set(v["answer"])) for v in sub.values())
        z = f"{OUT}/submission_{TAG}_n{n}.zip"
        json.dump(sub, open(f"{OUT}/submission.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f:
            f.write(f"{OUT}/submission.json", "submission.json")
        print(f"OK {z}")
    print("\nNỘP theo thứ tự n=2, n=1, n=3. Rổ ĐỔI thì ĐỈNH n DỜI (kết luận 3),")
    print("mà dev300 mù dưới 2 điểm -> ĐỪNG chọn n bằng dev. MỐC PHẢI VƯỢT: 0.9118 (submission_matchEmb_M20_n3)")